# add_style_chain.ipynb

Developer notebook for adding a new built-in style chain to the app.

## Workflow

1. **Setup** — locate repo, load both catalogs.
2. **Author the chain** — list the style steps and strengths.
3. **Validate** — abort if any referenced style is absent from `styles/catalog.json`.
4. **Install** — run the chain on `sample_images/arch.png` to generate a 256 px preview,
   write `style_chains/<id>/chain.yml` + `preview.jpg`, and update `style_chains/catalog.json`.
5. **Smoke-check** — reload catalog, confirm entry present, display preview.
6. **Commit** — `git add -A ; git commit -m 'feat: add <name> style chain'`

> Run from the `training/` directory **or** from the repo root.

## Step 1 — Setup

In [ ]:
import sys
from pathlib import Path

# Make sure the repo root is on sys.path so src/ is importable
# (works whether the CWD is training/ or the repo root)
_here = Path().resolve()
_repo = _here.parent if _here.name == 'training' else _here
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))

from training.add_style_chain_helper import setup

ctx = setup(repo_root=_repo)
print('\nReady.')

## Step 2 — Author the chain

Edit the variables below, then run the cell.

In [ ]:
# ── Chain metadata ─────────────────────────────────────────────────────────
CHAIN_NAME  = "My New Chain"          # display name, e.g. "Pastel"
CHAIN_DESC  = "Short description."    # shown as gallery tooltip
CHAIN_TAGS  = []                      # optional, e.g. ["warm", "painterly"]

# ── Steps: style names must match exactly the 'name' field in styles/catalog.json
STEPS = [
    {"style": "Ukiyo-e",      "strength": 150},
    {"style": "Cubism",       "strength": 80},
]

print(f'Chain : {CHAIN_NAME}')
for i, s in enumerate(STEPS, 1):
    print(f'  Step {i}: {s["style"]}  {s["strength"]}%')

## Step 3 — Validate styles

Abort if any referenced style is absent from the catalog.

In [ ]:
from training.add_style_chain_helper import validate_chain_styles

unknown = validate_chain_styles(STEPS, ctx.styles_catalog)
if unknown:
    raise ValueError(
        f'The following styles are not in styles/catalog.json:\n'
        + '\n'.join(f'  • {n}' for n in unknown)
    )
print('All styles valid ✓')

## Step 4 — Install chain

Generates the preview (applies the chain to `sample_images/arch.png`),
writes `style_chains/<id>/chain.yml` + `preview.jpg`,
and updates `style_chains/catalog.json`.

In [ ]:
from training.add_style_chain_helper import install_chain

chain_id = install_chain(
    steps=STEPS,
    chain_name=CHAIN_NAME,
    chain_desc=CHAIN_DESC,
    chain_tags=CHAIN_TAGS,
    chains_dir=ctx.chains_dir,
    chains_catalog_path=ctx.chains_catalog_path,
    chains_catalog=ctx.chains_catalog,
    existing_chain_ids=ctx.existing_chain_ids,
    content_image=ctx.content_image,
    repo_root=ctx.repo_root,
    preview_size=256,
)

## Step 5 — Smoke-check

Reload the catalog and display the preview thumbnail.

In [ ]:
import json
from IPython.display import display, Image as IPImage

with open(ctx.chains_catalog_path, encoding='utf-8') as fh:
    reloaded = json.load(fh)

entry = next((c for c in reloaded['chains'] if c['id'] == chain_id), None)
assert entry is not None, f"'{chain_id}' not found after reload!"

print('Catalog entry:')
print(json.dumps(entry, indent=2))

preview = ctx.repo_root / entry['preview_path']
if preview.exists():
    display(IPImage(filename=str(preview)))
else:
    print(f'WARNING: preview not found at {preview}')

## Step 6 — Commit

```bash
git add -A
git commit -m "feat: add <name> style chain"
```